# OnKith full-pipeline CPU benchmark

One notebook, any CPU. Run it on a laptop, run the identical file on the
Raspberry Pi, commit both result folders, and the numbers are comparable
because both runs scored the same committed rows with the same hash-verified
INT8 model.

**What it measures**

| Part | Question |
|---|---|
| A - full pipeline | How long does a real utterance take to travel through VAD -> STT -> transcript merge -> buffer -> privacy masking on this device? |
| B - privacy model | Does the accepted DeBERTa-v3-xsmall INT8 deployment path produce the expected quality here, and how fast and how large is it? |

Part B is the project's research contribution, so it is measured in detail.
Part A exists to show the deployment is feasible end to end; the STT model is a
pinned third-party component and only its timing matters here.

**How to run**

1. `git pull`
2. Open this notebook with the repository virtual environment as the kernel.
3. **Run All.**
4. Commit the new folder under `research_results/cpu_benchmarks/`.

Nothing has to be edited first. `SMOKE_TEST` in section 1 switches to a ~50-row
check that finishes in a couple of minutes; the committed default runs the full
2,000-row benchmark.

**Expected wall clock.** The full run is dominated by one ONNX inference per
row. A recent x86 laptop takes roughly 10-20 minutes; a Raspberry Pi 5 should
be planned for an hour or more. `SMOKE_TEST = True` first if you want to check
the environment before committing to that.

**Outputs.** Everything lands in
`research_results/cpu_benchmarks/<run_id>/`, machine-readable, plus a
`benchmark_summary.md` you can read directly. `Research_Paper_Figures.ipynb`
discovers those folders automatically and turns them into the deployment
figures.

**Privacy.** No source text is written to any output file. Predictions are
recorded as offsets, labels and hashes.

## 1. Repository, imports and configuration

In [ ]:
import csv
import hashlib
import json
import os
import platform
import statistics
import subprocess
import sys
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "FullPipeline_V3").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "FullPipeline_V3").is_dir():
    raise RuntimeError("Run this notebook from inside the OnKith repository.")

# The pipeline and the evaluation harness are imported, never reimplemented.
for extra in (REPO_ROOT / "FullPipeline_V3", REPO_ROOT / "model" / "privacy_evaluation"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

import numpy as np

try:
    import psutil
except ModuleNotFoundError:  # optional: only the memory section needs it
    psutil = None

print("repository :", REPO_ROOT)
print("python     :", sys.version.split()[0])
print("psutil     :", psutil.__version__ if psutil else "not installed (memory metrics reduced)")

In [ ]:
# =========================== CONFIGURATION ===========================
# The committed default is the real benchmark. Set SMOKE_TEST = True for a
# fast end-to-end check of imports, paths, model loading and file writing.

SMOKE_TEST = False          # True -> SMOKE_ROWS rows, a few minutes
SMOKE_ROWS = 50             # rows scored in smoke mode
BENCHMARK_ROWS = 2000       # rows scored in full mode (the whole committed subset)

WARMUP_RUNS = 3             # privacy-model warm-ups excluded from every latency stat
LATENCY_PROBE_ROWS = 60     # rows used for the fine-grained tokenize/infer/decode split

RUN_FULL_PIPELINE = True    # part A
PIPELINE_WARMUP_RUNS = 2
PIPELINE_LATENCY_RUNS = 20  # utterances pushed through the assembled pipeline

RUN_OOD_DEV = True          # 360-case out-of-distribution cross-check against the release number

RANDOM_SEED = 20260817      # only used for the synthetic VAD/STT waveform
DEVICE_LABEL = ""           # optional tag in the run id, e.g. "pi5" or "laptop"
ONNX_INTRA_OP_THREADS = None  # None keeps the production default

# =====================================================================

# Environment override so a quick check does not need an edit:
#   PowerShell : $env:ONKITH_BENCHMARK_SMOKE = "1"
#   bash       : export ONKITH_BENCHMARK_SMOKE=1
_override = os.environ.get("ONKITH_BENCHMARK_SMOKE")
if _override is not None:
    SMOKE_TEST = _override.strip() not in ("", "0", "false", "False")
DEVICE_LABEL = os.environ.get("ONKITH_BENCHMARK_DEVICE", DEVICE_LABEL)

MODE = "smoke" if SMOKE_TEST else "full"
TARGET_ROWS = SMOKE_ROWS if SMOKE_TEST else BENCHMARK_ROWS
if SMOKE_TEST:
    PIPELINE_LATENCY_RUNS = min(PIPELINE_LATENCY_RUNS, 5)
    LATENCY_PROBE_ROWS = min(LATENCY_PROBE_ROWS, 10)

STARTED_UTC = datetime.now(timezone.utc)


def _slug(value: str) -> str:
    """Filesystem- and URL-safe token; run ids become directory names."""

    kept = [character if character.isalnum() else "-" for character in value.strip().lower()]
    return "".join(kept).strip("-")[:24] or "unknown"


RUN_ID = "_".join(
    part
    for part in (
        STARTED_UTC.strftime("%Y%m%dT%H%M%SZ"),
        f"{_slug(platform.system())}-{_slug(platform.machine())}",
        MODE,
        _slug(DEVICE_LABEL) if DEVICE_LABEL else "",
    )
    if part
)

RESULTS_ROOT = REPO_ROOT / "research_results" / "cpu_benchmarks"
RUN_DIR = RESULTS_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

WARNINGS: list[str] = []       # collected and written into benchmark_summary.md
COMPLETED: dict[str, bool] = {}


def note(message: str) -> None:
    WARNINGS.append(message)
    print("WARNING:", message)


print(f"mode        : {MODE.upper()}  ({TARGET_ROWS} rows)")
print(f"run id      : {RUN_ID}")
print(f"output      : {RUN_DIR.relative_to(REPO_ROOT).as_posix()}")
print(f"started     : {STARTED_UTC.isoformat()}")

## 2. Hardware and runtime

Identification only. There is no separate code path for the Raspberry Pi: the
same INT8 model, the same tokenizer, the same production decoder and the same
metric code run everywhere, and any difference in the numbers is a property of
the hardware rather than of the software.

In [ ]:
def _read_first(*paths: str) -> str:
    for candidate in paths:
        try:
            text = Path(candidate).read_text(errors="ignore").strip("\x00").strip()
        except OSError:
            continue
        if text:
            return text
    return ""


def cpu_model_name() -> str:
    """Best available CPU name. platform.processor() is empty on most Linux ARM."""

    if platform.system() == "Linux":
        cpuinfo = _read_first("/proc/cpuinfo")
        fields = {}
        for line in cpuinfo.splitlines():
            if ":" in line:
                key, _, value = line.partition(":")
                fields.setdefault(key.strip().lower(), value.strip())
        for key in ("model name", "hardware", "cpu implementer", "processor"):
            if fields.get(key):
                return fields[key]
    if platform.system() == "Darwin":
        try:
            return subprocess.run(
                ["sysctl", "-n", "machdep.cpu.brand_string"],
                capture_output=True, text=True, timeout=5,
            ).stdout.strip()
        except (OSError, subprocess.SubprocessError):
            pass
    return platform.processor() or platform.machine() or "unknown"


def board_model() -> str:
    """Raspberry Pi and other device-tree boards announce themselves here."""

    return _read_first(
        "/proc/device-tree/model",
        "/sys/firmware/devicetree/base/model",
        "/sys/devices/virtual/dmi/id/product_name",
    )


import importlib.metadata  # noqa: E402
import onnxruntime as ort  # noqa: E402  - needed for the provider list


def installed(distribution: str):
    """Version without importing the package.

    ``import torch`` costs several hundred MiB of RSS, which would land in this
    notebook's own memory numbers and make them useless on a 4 GB Pi.
    """

    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


virtual_memory = psutil.virtual_memory() if psutil else None
board = board_model()
BASELINE_RSS = psutil.Process().memory_info().rss if psutil else None

SYSTEM_INFO = {
    "run_id": RUN_ID,
    "collected_utc": STARTED_UTC.isoformat(),
    "os": platform.system(),
    "platform": platform.platform(),
    "release": platform.release(),
    "architecture": platform.machine(),
    "process_architecture": platform.architecture()[0],
    "byte_order": sys.byteorder,
    "cpu_model": cpu_model_name(),
    "logical_cpus": os.cpu_count(),
    "physical_cpus": psutil.cpu_count(logical=False) if psutil else None,
    "board_model": board or None,
    "is_raspberry_pi": "raspberry pi" in board.lower(),
    "hostname": platform.node(),
    "total_ram_bytes": virtual_memory.total if virtual_memory else None,
    "available_ram_bytes": virtual_memory.available if virtual_memory else None,
    "baseline_process_rss_bytes": BASELINE_RSS,
    "python": platform.python_version(),
    "python_implementation": platform.python_implementation(),
    "numpy": np.__version__,
    "onnxruntime": ort.__version__,
    "onnxruntime_providers": list(ort.get_available_providers()),
    "transformers": installed("transformers"),
    "tokenizers": installed("tokenizers"),
    "torch": installed("torch"),
    "psutil": psutil.__version__ if psutil else None,
}


def _gib(value):
    return f"{value / 1024 ** 3:.2f} GiB" if value else "unknown"


print("Hardware / Runtime")
print("------------------")
print(f"Platform     : {SYSTEM_INFO['platform']}")
print(f"Architecture : {SYSTEM_INFO['architecture']} ({SYSTEM_INFO['process_architecture']} process)")
print(f"CPU          : {SYSTEM_INFO['cpu_model']}")
print(f"Logical CPUs : {SYSTEM_INFO['logical_cpus']}  (physical {SYSTEM_INFO['physical_cpus']})")
print(f"Board        : {SYSTEM_INFO['board_model'] or 'not reported'}")
print(f"RAM          : {_gib(SYSTEM_INFO['total_ram_bytes'])} total, {_gib(SYSTEM_INFO['available_ram_bytes'])} available")
print(f"Python       : {SYSTEM_INFO['python']} ({SYSTEM_INFO['python_implementation']})")
print(f"ONNX Runtime : {SYSTEM_INFO['onnxruntime']}  providers={SYSTEM_INFO['onnxruntime_providers']}")
print(f"transformers : {SYSTEM_INFO['transformers']}  numpy {SYSTEM_INFO['numpy']}  torch {SYSTEM_INFO['torch']}")
print(f"baseline RSS : {_gib(BASELINE_RSS)} before any model is loaded")

if SYSTEM_INFO["is_raspberry_pi"]:
    print("\nRaspberry Pi detected. Same model, same code, same rows as the workstation run.")

## 3. Artifact identity

The whole point of running this notebook twice is to compare two devices on the
*same* model. So before anything is measured, the packaged INT8 graph is
re-hashed and checked against three independent records:

* `config/live_config.json` - the hash the live pipeline refuses to start without;
* `models/deberta_pii/model_manifest.json` - the deployment package record;
* `accepted_int8_candidate.json` - the release gate, when the repository copy is present.

A mismatch aborts the privacy benchmark rather than producing numbers for an
unknown artifact.

In [ ]:
from integrated_pipeline.config import PipelineConfig  # noqa: E402

CONFIG = PipelineConfig(root=REPO_ROOT / "FullPipeline_V3")


def sha256_file(path, chunk_bytes: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()


INT8_PATH = CONFIG.pii_model_path
TOKENIZER_DIR = CONFIG.pii_tokenizer_path
LABEL_MAP_PATH = CONFIG.pii_label_map_path
DEPLOY_MANIFEST_PATH = INT8_PATH.parent / "model_manifest.json"

ARTIFACT_OK = True
artifact_files = {}
for label, path in (
    ("model_int8.onnx", INT8_PATH),
    ("tokenizer/tokenizer.json", TOKENIZER_DIR / "tokenizer.json"),
    ("tokenizer/tokenizer_config.json", TOKENIZER_DIR / "tokenizer_config.json"),
    ("config/model_v2_label_map.json", LABEL_MAP_PATH),
):
    if not path.is_file():
        note(f"missing deployment artifact: {path}")
        ARTIFACT_OK = False
        continue
    artifact_files[label] = {
        "path": path.relative_to(REPO_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }

label_map = json.loads(LABEL_MAP_PATH.read_text(encoding="utf-8"))
id2label = {int(key): str(value) for key, value in label_map["id2label"].items()}

pinned_sha = str(CONFIG.pii["sha256"])
measured_sha = artifact_files.get("model_int8.onnx", {}).get("sha256")
deploy_manifest = json.loads(DEPLOY_MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_sha = deploy_manifest["files"]["model_int8.onnx"]["sha256"]

checks = {
    "live_config.pii.sha256": pinned_sha == measured_sha,
    "model_manifest.files.model_int8.onnx.sha256": manifest_sha == measured_sha,
    "label_map.ontology_version": label_map["ontology_version"] == CONFIG.pii["ontology_version"],
    "label_map.num_labels": int(label_map["num_labels"]) == int(CONFIG.pii["label_count"]),
}

# The release gate lives in the model workspace. It is present in a full clone
# and its absence is not an error, only less corroboration.
ACCEPTED_CANDIDATE_PATH = (
    REPO_ROOT / "model" / "DeBERTa-v3-xsmall" / "artifacts" / "model_v2"
    / "metrics" / "accepted_int8_candidate.json"
)
accepted_candidate = None
if ACCEPTED_CANDIDATE_PATH.is_file():
    accepted_candidate = json.loads(ACCEPTED_CANDIDATE_PATH.read_text(encoding="utf-8"))
    checks["accepted_int8_candidate.model.int8_sha256"] = (
        accepted_candidate["model"]["int8_sha256"] == measured_sha
    )
    checks["accepted_int8_candidate.accepted"] = accepted_candidate.get("accepted") is True
else:
    note("accepted_int8_candidate.json not found; hash checked against the deployment records only")

FROZEN_OOD_PATH = ACCEPTED_CANDIDATE_PATH.parent / "evaluation_ood_final_int8.json"
frozen_report = (
    json.loads(FROZEN_OOD_PATH.read_text(encoding="utf-8")) if FROZEN_OOD_PATH.is_file() else None
)

RELEASE_METRICS = {}
if accepted_candidate:
    RELEASE_METRICS = accepted_candidate.get("evaluation", {}).get("int8_onnx", {})

ARTIFACT_MANIFEST = {
    "run_id": RUN_ID,
    "model": {
        "name": deploy_manifest.get("model"),
        "model_id": deploy_manifest.get("model_id"),
        "ontology_version": label_map["ontology_version"],
        "entity_types": int(label_map["num_entities"]),
        "label_count": int(label_map["num_labels"]),
        "precision": "int8",
        "provider": str(CONFIG.pii["provider"]),
        "context_limit": int(CONFIG.pii["context_limit"]),
        "overflow_stride": int(CONFIG.pii["overflow_stride"]),
        "decoder": "integrated_pipeline.model_v2_masker.ModelV2OnnxMasker",
    },
    "files": artifact_files,
    "identity_checks": checks,
    "identity_verified": all(checks.values()),
    "accepted_candidate_manifest": (
        ACCEPTED_CANDIDATE_PATH.relative_to(REPO_ROOT).as_posix()
        if accepted_candidate else None
    ),
    "accepted_candidate_manifest_sha256": (
        sha256_file(ACCEPTED_CANDIDATE_PATH) if accepted_candidate else None
    ),
    "frozen_ood_final_spent": bool(frozen_report),
}

ARTIFACT_OK = ARTIFACT_OK and ARTIFACT_MANIFEST["identity_verified"]

print(f"INT8 model   : {INT8_PATH.relative_to(REPO_ROOT).as_posix()}")
print(f"  size       : {INT8_PATH.stat().st_size:,} bytes ({INT8_PATH.stat().st_size / 1024 ** 2:.2f} MiB)")
print(f"  sha256     : {measured_sha}")
print(f"tokenizer    : {TOKENIZER_DIR.relative_to(REPO_ROOT).as_posix()}")
print(f"label map    : {label_map['num_entities']} entities / {label_map['num_labels']} BIO labels"
      f"  ({label_map['ontology_version']})")
print("\nidentity checks")
for name, passed in checks.items():
    print(f"  {'OK  ' if passed else 'FAIL'}  {name}")

if not ARTIFACT_OK:
    note("ARTIFACT IDENTITY FAILED - the privacy benchmark will be skipped")
else:
    print("\nThis is the accepted production artifact. Results are comparable across devices.")

## 4. Benchmark rows

`model/privacy_evaluation/fixtures/deployment_benchmark/` holds 2,000 rows
selected once from the held-out test split: every zero-PII row, a floor of
twelve rows for each entity type that has them, and a natural fill for the rest.
It is committed, so the Pi scores exactly these rows without rebuilding any
corpus. Smoke mode takes a prefix of the same file, and the file is stored in
`row_id` order, so a smoke prefix is the same prefix everywhere.

`ood_dev.jsonl` is scored separately as a cross-check: the release recorded a
number for it, so reproducing that number here proves the deployment path on
this device matches the one the release was measured on.

In [ ]:
from privacy_eval import deployment_benchmark as bench  # noqa: E402
from privacy_eval.dataset_metrics import evaluate_spans  # noqa: E402

benchmark_rows_all, SUBSET_MANIFEST = bench.load_subset()
BENCHMARK_SET = bench.take(benchmark_rows_all, TARGET_ROWS)

# A prefix of 50 rows can easily contain none of the four zero-PII rows, which
# would leave the hard-negative false-positive rate with no denominator. Add
# them back; they are a fixed, named set, so the smoke selection stays exact.
if len(BENCHMARK_SET) < len(benchmark_rows_all):
    present = {row["row_id"] for row in BENCHMARK_SET}
    BENCHMARK_SET += [
        row for row in benchmark_rows_all
        if not row["privacy_mask"] and row["row_id"] not in present
    ]

print(f"benchmark subset : {bench.SUBSET_PATH.relative_to(REPO_ROOT).as_posix()}")
print(f"  sha256         : {SUBSET_MANIFEST['fixture_sha256']}")
print(f"  available rows : {len(benchmark_rows_all):,}")
print(f"  scoring        : {len(BENCHMARK_SET):,} rows ({MODE} mode)")
coverage = SUBSET_MANIFEST["coverage"]
print(f"  entity types   : {coverage['entity_types_present']}")
print(f"  zero-PII rows  : {coverage['rows_without_pii']}")
print(f"  source         : {SUBSET_MANIFEST['source']['path']} ({SUBSET_MANIFEST['source']['view']} split)")

OOD_ROWS = []
if RUN_OOD_DEV:
    from privacy_eval.schema import load_cases  # noqa: E402

    ood_cases = load_cases(REPO_ROOT / "model" / "privacy_evaluation" / "fixtures" / "ood_dev.jsonl")
    OOD_ROWS = [
        {
            "row_id": case.case_id,
            "source_text": case.text,
            "privacy_mask": [
                {"start": entity.start, "end": entity.end,
                 "label": entity.entity_type, "value": entity.text}
                for entity in case.entities
            ],
            "category": case.category,
            "tags": list(case.tags),
        }
        for case in ood_cases
    ]
    OOD_ROWS.sort(key=lambda row: row["row_id"])
    if SMOKE_TEST:
        # A prefix would be one category; a stride keeps every category and the
        # hard negatives, so the smoke number is at least the right shape. It is
        # still a subset, so it is not expected to equal the release figure.
        OOD_ROWS = OOD_ROWS[:: max(1, len(OOD_ROWS) // 40)][:40]
    print(f"\nOOD dev cases    : {len(OOD_ROWS)}")

# Deterministic spoken-style utterances for the assembled pipeline, taken from
# the committed OOD dev fixture so every device pushes the same words through.
from privacy_eval.schema import load_cases as _load_cases  # noqa: E402

_all_dev = sorted(
    _load_cases(REPO_ROOT / "model" / "privacy_evaluation" / "fixtures" / "ood_dev.jsonl"),
    key=lambda case: case.case_id,
)
PIPELINE_UTTERANCES = [
    case.text
    for case in _all_dev
    if "speech_like" in case.tags and 40 <= len(case.text) <= 220
][:PIPELINE_LATENCY_RUNS]
print(f"pipeline inputs  : {len(PIPELINE_UTTERANCES)} spoken-style utterances")

## 5. Load the production privacy path

`PIIMaskerAdapter` is the class the live pipeline uses. It verifies the artifact
hash, loads the packaged tokenizer offline, builds the ONNX session, and applies
the structured recognizer and the mask policy on top of the neural decode. The
model-only decoder underneath it, `ModelV2OnnxMasker`, is the same object the
release evaluation was measured through.

Loading and the first inferences are timed separately and excluded from every
latency statistic below.

In [ ]:
from integrated_pipeline.model_v2_masker import ModelV2OnnxMasker  # noqa: E402
from integrated_pipeline.pii_adapter import PIIMaskerAdapter  # noqa: E402

WARMUP_TEXT = (
    "Hi, my name is Sarah Walusiak, my number is 555 0147 and my email is "
    "sarah.walusiak@example.org. Remind me about the appointment on 12 March."
)

ADAPTER = None
LOAD_SECONDS = None
RSS_AFTER_LOAD = None
WARMUP_LATENCIES: list[float] = []

if ARTIFACT_OK:
    ADAPTER = PIIMaskerAdapter(CONFIG)
    started = time.perf_counter()
    ADAPTER.load()
    LOAD_SECONDS = time.perf_counter() - started
    RSS_AFTER_LOAD = psutil.Process().memory_info().rss if psutil else None
    print(f"model load          : {LOAD_SECONDS:.2f} s")
    if RSS_AFTER_LOAD and BASELINE_RSS:
        print(f"RSS after load      : {RSS_AFTER_LOAD / 1024 ** 2:.1f} MiB "
              f"(+{(RSS_AFTER_LOAD - BASELINE_RSS) / 1024 ** 2:.1f} MiB over baseline)")

    for index in range(WARMUP_RUNS):
        started = time.perf_counter()
        ADAPTER.mask_text(WARMUP_TEXT)
        WARMUP_LATENCIES.append(time.perf_counter() - started)
    print("warm-up (excluded)  : " + ", ".join(f"{value * 1000:.1f} ms" for value in WARMUP_LATENCIES))

    demo = ADAPTER.mask_text(WARMUP_TEXT)
    print("\nproduction masking, one example:")
    print(" ", demo.masked_text)
    print(f"  {len(demo.entities)} entities, "
          f"{demo.raw_model_output['window_count']} window(s), "
          f"{demo.raw_model_output['label_count']} labels")
else:
    note("privacy model not loaded because artifact identity failed")

## 6. Privacy-model stage latency

Where the time actually goes inside one masking call. The coarse split
(neural decode / structured regex / span merge) is read straight out of the
production instrumentation in `PIIMaskerAdapter.mask_text`, so it is the real
deployment path and not a re-timed copy of it.

The finer tokenize-versus-infer split needs one extra call per probe row, so it
runs on a small sample and is reported separately.

In [ ]:
def summarise(samples, unit_scale: float = 1000.0) -> dict:
    """count/mean/std/min/max/median/P90/P95/P99, in milliseconds by default."""

    values = sorted(float(sample) * unit_scale for sample in samples)
    if not values:
        return {"count": 0}

    def percentile(fraction: float) -> float:
        if len(values) == 1:
            return values[0]
        position = fraction * (len(values) - 1)
        low = int(position)
        high = min(low + 1, len(values) - 1)
        return values[low] + (values[high] - values[low]) * (position - low)

    return {
        "count": len(values),
        "mean_ms": statistics.fmean(values),
        "std_ms": statistics.pstdev(values) if len(values) > 1 else 0.0,
        "min_ms": values[0],
        "max_ms": values[-1],
        "p50_ms": percentile(0.50),
        "p90_ms": percentile(0.90),
        "p95_ms": percentile(0.95),
        "p99_ms": percentile(0.99),
        "total_ms": sum(values),
    }


def print_summary(title: str, summary: dict) -> None:
    if not summary.get("count"):
        print(f"{title}: no samples")
        return
    print(
        f"{title:<28} n={summary['count']:<6} mean {summary['mean_ms']:>9.2f}  "
        f"p50 {summary['p50_ms']:>9.2f}  p90 {summary['p90_ms']:>9.2f}  "
        f"p95 {summary['p95_ms']:>9.2f}  p99 {summary['p99_ms']:>9.2f}  max {summary['max_ms']:>9.2f}  (ms)"
    )


TOKENIZE_PROBE = {}
if ADAPTER is not None:
    masker: ModelV2OnnxMasker = ADAPTER._masker  # the object the adapter already loaded
    probe_rows = BENCHMARK_SET[:LATENCY_PROBE_ROWS]
    tokenize_seconds, decode_seconds, structured_seconds, token_counts = [], [], [], []

    for row in probe_rows:
        text = row["source_text"]
        started = time.perf_counter()
        _, windows = masker._windows(text)
        tokenize_seconds.append(time.perf_counter() - started)

        started = time.perf_counter()
        decoding = masker.decode(text)
        decode_seconds.append(time.perf_counter() - started)
        token_counts.append(decoding.token_count)

        started = time.perf_counter()
        ADAPTER._structured.find_spans(text)
        structured_seconds.append(time.perf_counter() - started)

    inference_seconds = [
        max(0.0, decode - tokenize) for decode, tokenize in zip(decode_seconds, tokenize_seconds)
    ]
    TOKENIZE_PROBE = {
        "rows": len(probe_rows),
        "note": (
            "Tokenization is measured directly; 'inference_and_consolidation' is "
            "decode() minus tokenization, because the ONNX call and the word "
            "consolidation share one pass in the production decoder."
        ),
        "tokenization": summarise(tokenize_seconds),
        "decode_total": summarise(decode_seconds),
        "inference_and_consolidation": summarise(inference_seconds),
        "structured_recognizer": summarise(structured_seconds),
        "source_tokens": {
            "mean": statistics.fmean(token_counts) if token_counts else 0,
            "min": min(token_counts) if token_counts else 0,
            "max": max(token_counts) if token_counts else 0,
        },
    }
    print(f"fine-grained probe over {len(probe_rows)} rows "
          f"(mean {TOKENIZE_PROBE['source_tokens']['mean']:.0f} source tokens)\n")
    print_summary("tokenize + window", TOKENIZE_PROBE["tokenization"])
    print_summary("ONNX + consolidate", TOKENIZE_PROBE["inference_and_consolidation"])
    print_summary("decode() total", TOKENIZE_PROBE["decode_total"])
    print_summary("structured regex", TOKENIZE_PROBE["structured_recognizer"])

## 7. Deployment evaluation on the held-out subset

One pass. Every row goes through the production masker once, and that single
pass produces the latency samples, the quality metrics and the parity record.

Two views come out of it:

* **production** - every entity the deployed stack detects, neural plus the
  structured regex layer. This is what the device actually does.
* **model-attributed** - the subset of those entities the model produced. The
  release evaluation measured the model alone, so this is the closer comparison
  to the published numbers; it is not identical, because span merging happens
  before the attribution is read.

In [ ]:
def spans_of(entities, model_only: bool = False):
    return [
        {"start": int(entity["start"]), "end": int(entity["end"]),
         "label": str(entity["entity_type"])}
        for entity in entities
        if not model_only or not str(entity.get("source", "")).startswith("structured")
    ]


def span_digest(spans) -> str:
    payload = ";".join(f"{s['start']}-{s['end']}:{s['label']}" for s in spans)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]


def rss_bytes():
    return psutil.Process().memory_info().rss if psutil else None


EVAL_ROWS = []
PRODUCTION_PREDICTIONS = []
MODEL_PREDICTIONS = []
PARITY_RECORDS = []
ROW_LATENCY = []
STAGE_LATENCY = {"neural_decode": [], "structured": [], "span_merge": []}
PEAK_RSS = rss_bytes() or 0
FAILURES = []

if ADAPTER is not None:
    print(f"scoring {len(BENCHMARK_SET):,} rows ...")
    loop_started = time.perf_counter()
    for index, row in enumerate(BENCHMARK_SET, start=1):
        text = row["source_text"]
        started = time.perf_counter()
        try:
            result = ADAPTER.mask_text(text)
        except Exception as exc:  # a failed row must not lose the whole benchmark
            FAILURES.append({"row_id": row["row_id"], "error": f"{type(exc).__name__}: {exc}"})
            continue
        elapsed = time.perf_counter() - started

        production = spans_of(result.entities)
        model_only = spans_of(result.entities, model_only=True)
        raw = result.raw_model_output

        EVAL_ROWS.append(row)
        PRODUCTION_PREDICTIONS.append(production)
        MODEL_PREDICTIONS.append(model_only)
        ROW_LATENCY.append(elapsed)
        STAGE_LATENCY["neural_decode"].append(float(raw["pii_model_latency_seconds"]))
        STAGE_LATENCY["structured"].append(float(raw["structured_latency_seconds"]))
        STAGE_LATENCY["span_merge"].append(float(raw["span_merger_latency_seconds"]))

        # Offsets, labels and hashes only. No source text leaves this loop.
        PARITY_RECORDS.append({
            "row_id": row["row_id"],
            "characters": len(text),
            "windows": int(raw["window_count"]),
            "bio_repairs": int(raw["bio_repairs"]),
            "production_spans": [[s["start"], s["end"], s["label"]] for s in production],
            "production_digest": span_digest(production),
            "model_digest": span_digest(model_only),
            "masked_text_sha256": hashlib.sha256(
                result.masked_text.encode("utf-8")
            ).hexdigest()[:16],
            "latency_ms": elapsed * 1000.0,
        })

        if index % 25 == 0 or index == len(BENCHMARK_SET):
            current = rss_bytes()
            if current:
                PEAK_RSS = max(PEAK_RSS, current)
        if index % 250 == 0:
            rate = index / (time.perf_counter() - loop_started)
            print(f"  {index:>6,} / {len(BENCHMARK_SET):,}  ({rate:.1f} rows/s)")

    EVAL_SECONDS = time.perf_counter() - loop_started
    print(f"\nscored {len(EVAL_ROWS):,} rows in {EVAL_SECONDS:.1f} s "
          f"({len(EVAL_ROWS) / EVAL_SECONDS:.2f} rows/s)")
    if FAILURES:
        note(f"{len(FAILURES)} row(s) raised during masking; see evaluation_metrics.json")
    COMPLETED["deberta_benchmark"] = True
else:
    EVAL_SECONDS = 0.0
    COMPLETED["deberta_benchmark"] = False

In [ ]:
DEBERTA_LATENCY = {}
EVALUATION = {}
PER_ENTITY = []

if EVAL_ROWS:
    DEBERTA_LATENCY = {
        "rows": len(ROW_LATENCY),
        "mode": MODE,
        "load_seconds": LOAD_SECONDS,
        "warmup_runs": WARMUP_RUNS,
        "warmup_ms": [value * 1000.0 for value in WARMUP_LATENCIES],
        "wall_clock_seconds": EVAL_SECONDS,
        "throughput_rows_per_second": len(EVAL_ROWS) / EVAL_SECONDS if EVAL_SECONDS else None,
        "end_to_end_masking_call": summarise(ROW_LATENCY),
        "stages": {name: summarise(values) for name, values in STAGE_LATENCY.items()},
        "fine_grained_probe": TOKENIZE_PROBE,
    }

    production_metrics = evaluate_spans(EVAL_ROWS, PRODUCTION_PREDICTIONS)
    model_metrics = evaluate_spans(EVAL_ROWS, MODEL_PREDICTIONS)
    EVALUATION = {
        "run_id": RUN_ID,
        "mode": MODE,
        "view": "deployment_benchmark_subset",
        "rows": len(EVAL_ROWS),
        "subset_sha256": SUBSET_MANIFEST["fixture_sha256"],
        "int8_sha256": measured_sha,
        "failures": FAILURES,
        "production_path": production_metrics,
        "model_attributed": model_metrics,
    }

    for label, entry in sorted(production_metrics["per_label"].items()):
        PER_ENTITY.append({
            "view": "deployment_benchmark_subset",
            "entity": label,
            "precision": entry["precision"],
            "recall": entry["recall"],
            "f1": entry["f1"],
            "support": entry["support"],
            "predicted": entry["predicted"],
            "true_positives": entry["true_positives"],
        })

    print("Deployment benchmark - production path")
    print(f"  rows                              {production_metrics['rows']:,}")
    print(f"  typed entity F1                   {production_metrics['typed_entity']['f1']:.4f}")
    print(f"  typed entity precision            {production_metrics['typed_entity']['precision']:.4f}")
    print(f"  typed entity recall               {production_metrics['typed_entity']['recall']:.4f}")
    print(f"  binary entity F1                  {production_metrics['binary_entity']['f1']:.4f}")
    print(f"  private character recall          {production_metrics['private_character_recall']:.4f}")
    print(f"  private character leakage         {production_metrics['private_character_leakage']:.4f}")
    print(f"  non-private overmasking rate      {production_metrics['non_private_overmasking_rate']:.4f}")
    print(f"  hard-negative false positive rate {production_metrics['hard_negative_false_positive_rate']:.4f}"
          f"  ({production_metrics['hard_negative_rows']} zero-PII rows)")
    print(f"\n  model-attributed typed F1         {model_metrics['typed_entity']['f1']:.4f}")

    print("\nlatency")
    print_summary("full masking call", DEBERTA_LATENCY["end_to_end_masking_call"])
    for name in ("neural_decode", "structured", "span_merge"):
        print_summary(name, DEBERTA_LATENCY["stages"][name])

    weakest = sorted(
        (entry for entry in PER_ENTITY if entry["support"] >= 5),
        key=lambda entry: entry["f1"],
    )[:5]
    print("\nweakest entity types with at least 5 gold spans:")
    for entry in weakest:
        print(f"  {entry['entity']:20} F1 {entry['f1']:.4f}  (support {entry['support']})")

## 8. Out-of-distribution cross-check

`ood_dev` is the one view where the release recorded a number for this exact
artifact through the model-only path. Reproducing it here is the strongest
available evidence that this device runs the same model the release measured -
stronger than any self-consistency check, because the target was written down
before this notebook existed.

In [ ]:
OOD_RESULT = {}
if OOD_ROWS and ADAPTER is not None:
    # Model-only, exactly as the release measured it: no structured recognizer.
    model_only_masker = ModelV2OnnxMasker(
        INT8_PATH, TOKENIZER_DIR, id2label,
        context_limit=int(CONFIG.pii["context_limit"]),
        overflow_stride=int(CONFIG.pii["overflow_stride"]),
        intra_op_threads=ONNX_INTRA_OP_THREADS,
    )
    model_only_masker.load()
    model_only_masker.mask_text(WARMUP_TEXT)  # warm-up, excluded

    started = time.perf_counter()
    ood_predictions, ood_latency = [], []
    for row in OOD_ROWS:
        row_started = time.perf_counter()
        result = model_only_masker.mask_text(row["source_text"])
        ood_latency.append(time.perf_counter() - row_started)
        ood_predictions.append([
            {"start": int(entity["start"]), "end": int(entity["end"]),
             "label": str(entity["entity_type"])}
            for entity in result.entities
        ])
    ood_seconds = time.perf_counter() - started
    ood_metrics = evaluate_spans(OOD_ROWS, ood_predictions)

    released = RELEASE_METRICS.get("ood_dev", {})
    delta = (
        ood_metrics["typed_entity"]["f1"] - released["typed_entity_f1"]
        if released else None
    )
    OOD_RESULT = {
        "rows": len(OOD_ROWS),
        "complete_fixture": len(OOD_ROWS) == 360,
        "seconds": ood_seconds,
        "metrics": ood_metrics,
        "latency": summarise(ood_latency),
        "release_reference": released or None,
        "typed_f1_delta_vs_release": delta,
        "parity": [
            {"case_id": row["row_id"],
             "digest": span_digest(prediction),
             "category": row.get("category")}
            for row, prediction in zip(OOD_ROWS, ood_predictions)
        ],
    }

    print(f"ood_dev, model-only path, {len(OOD_ROWS)} cases in {ood_seconds:.1f} s")
    print(f"  typed entity F1                   {ood_metrics['typed_entity']['f1']:.4f}")
    print(f"  private character recall          {ood_metrics['private_character_recall']:.4f}")
    print(f"  hard-negative false positive rate {ood_metrics['hard_negative_false_positive_rate']:.4f}")
    if released:
        print(f"\n  release recorded typed F1         {released['typed_entity_f1']:.4f}")
        print(f"  delta                             {delta:+.4f}")
        if len(OOD_ROWS) != 360:
            print("  (smoke mode scores a subset, so a delta here is expected)")
        elif abs(delta) > 1e-4:
            note(f"ood_dev typed F1 differs from the release number by {delta:+.4f}")
        else:
            print("  the deployment path reproduces the release number exactly")
    COMPLETED["ood_dev_crosscheck"] = True
else:
    COMPLETED["ood_dev_crosscheck"] = False
    if RUN_OOD_DEV and ADAPTER is None:
        note("ood_dev cross-check skipped: privacy model unavailable")

## 9. Full pipeline (part A)

The assembled deployment path, stage by stage, with the repository's own
components:

```text
Silero VAD -> Canary STT -> transcript merge -> wake-word normalisation
   -> inactivity buffer -> privacy masking -> spoken-digit rendering
```

Stages that cannot run on this machine are reported as unavailable with the
reason, and the rest still run. Nothing is mocked: a missing stage produces no
number rather than a fake one.

SQLite storage is deliberately outside the measured path. `SQLiteTranscriptStore`
writes from a background thread behind a queue, so it does not sit in the
latency a speaker experiences.

In [ ]:
def deterministic_speech(seconds: float = 3.0, sample_rate: int = 16_000) -> np.ndarray:
    """Speech-shaped audio from a fixed seed: identical bytes on every device.

    Real recordings are not committed to this repository, and a fixed synthetic
    waveform is better for a *timing* comparison anyway - VAD and STT cost
    depends on length and energy structure, both of which are pinned here.
    """

    generator = np.random.default_rng(RANDOM_SEED)
    samples = int(seconds * sample_rate)
    time_axis = np.arange(samples, dtype=np.float64) / sample_rate
    # Voiced-ish carrier with formant-like harmonics, amplitude modulated at a
    # syllable rate, plus a little breath noise.
    carrier = sum(
        np.sin(2 * np.pi * frequency * time_axis) / weight
        for frequency, weight in ((120.0, 1.0), (360.0, 2.0), (720.0, 3.0), (1_450.0, 5.0))
    )
    envelope = 0.5 * (1.0 + np.sin(2 * np.pi * 3.1 * time_axis))
    noise = generator.normal(0.0, 0.02, samples)
    waveform = 0.25 * carrier * envelope + noise
    silence = int(0.35 * sample_rate)
    waveform[:silence] = noise[:silence] * 0.1
    waveform[-silence:] = noise[-silence:] * 0.1
    return np.clip(waveform, -1.0, 1.0).astype(np.float32)


PIPELINE_STAGES = {}
PIPELINE_ROWS = []
PIPELINE_END_TO_END = []
AUDIO = deterministic_speech()
AUDIO_SECONDS = len(AUDIO) / 16_000


def record_stage(name: str, samples, *, available: bool = True, reason: str = "",
                 extra: dict | None = None) -> None:
    entry = {"available": available, "reason": reason}
    if available:
        entry.update(summarise(samples))
    if extra:
        entry.update(extra)
    PIPELINE_STAGES[name] = entry

In [ ]:
# ---- Stage 1: Silero VAD -----------------------------------------------------
if RUN_FULL_PIPELINE:
    try:
        from integrated_pipeline.silero_vad import SileroOnnxVAD

        vad = SileroOnnxVAD(CONFIG)
        vad.load()
        frame_samples = int(CONFIG.silero["frame_samples"])
        frames = [
            AUDIO[start:start + frame_samples]
            for start in range(0, len(AUDIO) - frame_samples + 1, frame_samples)
        ]
        for _ in range(PIPELINE_WARMUP_RUNS):
            vad.reset()
            for frame in frames:
                vad.probability(frame)

        vad_seconds, speech_frames = [], 0
        for _ in range(max(3, PIPELINE_LATENCY_RUNS // 4)):
            vad.reset()
            started = time.perf_counter()
            probabilities = [vad.probability(frame) for frame in frames]
            vad_seconds.append(time.perf_counter() - started)
            speech_frames = sum(1 for value in probabilities if value >= float(CONFIG.silero["threshold"]))
        vad.close()

        per_frame = [value / len(frames) for value in vad_seconds]
        record_stage(
            "vad_silero", vad_seconds,
            extra={
                "audio_seconds": AUDIO_SECONDS,
                "frames_per_pass": len(frames),
                "speech_frames_last_pass": speech_frames,
                "per_frame_ms": summarise(per_frame),
                "real_time_factor": statistics.fmean(vad_seconds) / AUDIO_SECONDS,
            },
        )
        print(f"VAD  : {len(frames)} frames over {AUDIO_SECONDS:.1f} s audio, "
              f"mean {statistics.fmean(vad_seconds) * 1000:.1f} ms per pass "
              f"(RTF {PIPELINE_STAGES['vad_silero']['real_time_factor']:.4f})")
    except Exception as exc:
        record_stage("vad_silero", [], available=False,
                     reason=f"{type(exc).__name__}: {exc}")
        note(f"VAD stage unavailable: {type(exc).__name__}: {exc}")
else:
    record_stage("vad_silero", [], available=False, reason="RUN_FULL_PIPELINE is False")

In [ ]:
# ---- Stage 2: Canary STT -----------------------------------------------------
STT_TEXT = None
if RUN_FULL_PIPELINE:
    try:
        from integrated_pipeline.canary_adapter import CanaryAdapter, describe_nemo_backend
        from integrated_pipeline.schemas import CanarySegment

        backend = describe_nemo_backend()
        artifact = CONFIG.canary_model_path
        if not backend.get("available"):
            raise RuntimeError(
                "NeMo backend unavailable: " + "; ".join(backend.get("errors", ["unknown"]))
            )
        if not artifact.is_file():
            raise FileNotFoundError(
                f"{artifact.name} is not present. Fetch it with "
                "`python FullPipeline_V3/scripts/acquire_canary.py`."
            )

        canary = CanaryAdapter(CONFIG)
        canary.load()

        def make_segment(index: int) -> CanarySegment:
            return CanarySegment(
                session_id="benchmark", utterance_id="benchmark-utterance",
                segment_id=f"benchmark-segment-{index}", utterance_sequence_number=index,
                segment_index=index, samples=AUDIO, sample_rate=16_000,
                segment_start_sample=0, segment_end_sample=len(AUDIO),
                decision_sample=len(AUDIO), last_speech_end_sample=len(AUDIO),
                forced_boundary_sample=None, endpoint_reason="silence",
                is_final_segment_of_utterance=True, reliable=True,
                dropped_audio_chunks=0, recording_closed_monotonic=0.0,
                audio_started_monotonic=0.0,
            )

        for index in range(PIPELINE_WARMUP_RUNS):
            canary.transcribe_segment(make_segment(index))

        stt_seconds = []
        for index in range(max(3, PIPELINE_LATENCY_RUNS // 4)):
            started = time.perf_counter()
            transcript = canary.transcribe_segment(make_segment(100 + index))
            stt_seconds.append(time.perf_counter() - started)
            STT_TEXT = getattr(transcript, "text", None)
        canary.close()

        record_stage("stt_canary", stt_seconds, extra={
            "audio_seconds": AUDIO_SECONDS,
            "real_time_factor": statistics.fmean(stt_seconds) / AUDIO_SECONDS,
            "model": CONFIG.canary["model_id"],
        })
        print(f"STT  : mean {statistics.fmean(stt_seconds) * 1000:.0f} ms for {AUDIO_SECONDS:.1f} s "
              f"(RTF {PIPELINE_STAGES['stt_canary']['real_time_factor']:.3f})")
    except Exception as exc:
        reason = f"{type(exc).__name__}: {exc}"
        record_stage("stt_canary", [], available=False, reason=reason)
        note(f"STT stage unavailable - {reason}")
        print("STT  : unavailable, the remaining stages still run")
else:
    record_stage("stt_canary", [], available=False, reason="RUN_FULL_PIPELINE is False")

In [ ]:
# ---- Stages 3-7: text path ---------------------------------------------------
if RUN_FULL_PIPELINE and PIPELINE_UTTERANCES:
    from app.digit_normalizer import normalize_spoken_digits
    from app.wake_word_normalizer import normalize_wake_word
    from integrated_pipeline.inactivity_buffer import InactivityTranscriptBuffer
    from integrated_pipeline.schemas import TranscriptFragment
    from integrated_pipeline.transcript_merger import merge_segment_transcripts

    def split_into_segments(text: str) -> list[str]:
        """Two overlapping ASR segments, the shape the merger is built for."""

        words = text.split()
        if len(words) < 8:
            return [text]
        cut = len(words) // 2
        overlap = min(4, cut)
        return [" ".join(words[: cut + overlap]), " ".join(words[cut:])]

    merge_seconds, wake_seconds, buffer_seconds, mask_seconds, digit_seconds = [], [], [], [], []
    end_to_end = []

    for _ in range(PIPELINE_WARMUP_RUNS):
        for text in PIPELINE_UTTERANCES[:2]:
            merged = merge_segment_transcripts(split_into_segments(text))
            normalize_wake_word(merged.text)
            if ADAPTER is not None:
                ADAPTER.mask_text(merged.text)

    for index, text in enumerate(PIPELINE_UTTERANCES):
        segments = split_into_segments(text)
        utterance_started = time.perf_counter()

        started = time.perf_counter()
        merged = merge_segment_transcripts(segments)
        merge_seconds.append(time.perf_counter() - started)

        started = time.perf_counter()
        wake_text, _variant = normalize_wake_word(merged.text)
        wake_seconds.append(time.perf_counter() - started)

        started = time.perf_counter()
        buffer = InactivityTranscriptBuffer(
            inactivity_seconds=CONFIG.buffer_inactivity_seconds,
            maximum_tokens=CONFIG.maximum_buffer_tokens,
            maximum_duration_seconds=CONFIG.maximum_buffer_duration_seconds,
        )
        buffer.add_fragment(
            TranscriptFragment(
                fragment_id=f"benchmark-{index}",
                text=wake_text,
                arrival_time_seconds=0.0,
                source_audio_id=f"benchmark-audio-{index}",
                session_id="benchmark",
                batch_id=f"benchmark-{index}",
                sequence_number=index,
            )
        )
        flushed = buffer.flush(0.0, reason="manual")
        buffer_seconds.append(time.perf_counter() - started)
        buffered_text = flushed.text if flushed else wake_text

        masked_text = buffered_text
        if ADAPTER is not None:
            started = time.perf_counter()
            masked = ADAPTER.mask_text(buffered_text)
            mask_seconds.append(time.perf_counter() - started)
            masked_text = masked.masked_text

        started = time.perf_counter()
        normalize_spoken_digits(masked_text)
        digit_seconds.append(time.perf_counter() - started)

        total = time.perf_counter() - utterance_started
        end_to_end.append(total)
        PIPELINE_ROWS.append({
            "utterance_index": index,
            "characters": len(text),
            "words": len(text.split()),
            "transcript_merge_ms": merge_seconds[-1] * 1000.0,
            "wake_word_ms": wake_seconds[-1] * 1000.0,
            "inactivity_buffer_ms": buffer_seconds[-1] * 1000.0,
            "privacy_masking_ms": (mask_seconds[-1] * 1000.0) if mask_seconds else None,
            "digit_normalisation_ms": digit_seconds[-1] * 1000.0,
            "text_path_total_ms": total * 1000.0,
        })

    record_stage("transcript_merge", merge_seconds)
    record_stage("wake_word_normalisation", wake_seconds)
    record_stage("inactivity_buffer", buffer_seconds)
    if mask_seconds:
        record_stage("privacy_masking", mask_seconds)
    else:
        record_stage("privacy_masking", [], available=False,
                     reason="privacy model unavailable")
    record_stage("digit_normalisation", digit_seconds)
    PIPELINE_END_TO_END = end_to_end

    print(f"text path over {len(PIPELINE_UTTERANCES)} utterances\n")
    for name in ("transcript_merge", "wake_word_normalisation", "inactivity_buffer",
                 "privacy_masking", "digit_normalisation"):
        if PIPELINE_STAGES[name]["available"]:
            print_summary(name, PIPELINE_STAGES[name])
    print_summary("text path end to end", summarise(end_to_end))
    COMPLETED["pipeline_text_path"] = True
else:
    COMPLETED["pipeline_text_path"] = False
    if RUN_FULL_PIPELINE:
        note("pipeline text path skipped: no deterministic utterances resolved")

In [ ]:
# ---- Assembled end-to-end estimate -------------------------------------------
available_audio = [
    name for name in ("vad_silero", "stt_canary") if PIPELINE_STAGES.get(name, {}).get("available")
]
PIPELINE_SUMMARY = {
    "run_id": RUN_ID,
    "mode": MODE,
    "audio_seconds": AUDIO_SECONDS,
    "utterances": len(PIPELINE_UTTERANCES),
    "stages": PIPELINE_STAGES,
    "text_path_end_to_end": summarise(PIPELINE_END_TO_END) if PIPELINE_END_TO_END else {"count": 0},
    "complete": bool(PIPELINE_END_TO_END) and len(available_audio) == 2,
    "audio_stages_available": available_audio,
}
COMPLETED["full_pipeline"] = PIPELINE_SUMMARY["complete"]

if PIPELINE_END_TO_END:
    audio_mean_ms = sum(
        PIPELINE_STAGES[name]["mean_ms"] for name in available_audio
    )
    PIPELINE_SUMMARY["full_utterance_estimate_ms"] = (
        audio_mean_ms + PIPELINE_SUMMARY["text_path_end_to_end"]["mean_ms"]
    )
    PIPELINE_SUMMARY["full_utterance_estimate_note"] = (
        "Mean of the available audio stages plus the measured text path. "
        f"Audio stages included: {available_audio or 'none'}."
    )
    print(f"\nassembled estimate for one {AUDIO_SECONDS:.1f} s utterance: "
          f"{PIPELINE_SUMMARY['full_utterance_estimate_ms']:.1f} ms")
    print(f"  audio stages included: {available_audio or 'none'}")
    if not PIPELINE_SUMMARY["complete"]:
        print("  incomplete: at least one audio stage could not run on this machine")

## 10. Memory

In [ ]:
MEMORY = {
    "run_id": RUN_ID,
    "psutil_available": bool(psutil),
    "baseline_rss_bytes": BASELINE_RSS,
    "model_file_bytes": INT8_PATH.stat().st_size if INT8_PATH.is_file() else None,
    "tokenizer_bytes": (TOKENIZER_DIR / "tokenizer.json").stat().st_size
    if (TOKENIZER_DIR / "tokenizer.json").is_file() else None,
}

if psutil:
    process = psutil.Process()
    memory_info = process.memory_info()
    virtual = psutil.virtual_memory()
    MEMORY.update({
        "process_rss_bytes": memory_info.rss,
        "process_vms_bytes": memory_info.vms,
        "process_peak_rss_bytes": max(PEAK_RSS, memory_info.rss),
        "peak_rss_source": "sampled during the evaluation loop",
        "system_total_bytes": virtual.total,
        "system_available_bytes": virtual.available,
        "system_used_percent": virtual.percent,
    })
    # Platform-native peak, where the OS keeps one.
    native_peak = getattr(memory_info, "peak_wset", None)
    if native_peak is None and platform.system() != "Windows":
        try:
            import resource

            usage = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            # Linux reports KiB, macOS reports bytes.
            native_peak = usage * 1024 if platform.system() == "Linux" else usage
        except Exception:
            native_peak = None
    if native_peak:
        MEMORY["process_peak_rss_bytes_os"] = int(native_peak)
        MEMORY["process_peak_rss_bytes"] = max(
            MEMORY["process_peak_rss_bytes"], int(native_peak)
        )

    if BASELINE_RSS and RSS_AFTER_LOAD:
        MEMORY["rss_after_model_load_bytes"] = RSS_AFTER_LOAD
        MEMORY["privacy_stack_resident_bytes"] = max(0, RSS_AFTER_LOAD - BASELINE_RSS)
        MEMORY["privacy_stack_resident_note"] = (
            "RSS immediately after PIIMaskerAdapter.load() minus the RSS before it. "
            "This is the whole privacy stack, not the graph alone: the ONNX session, "
            "the SentencePiece tokenizer, and the transformers import chain that "
            "AutoTokenizer pulls in. On a 4 GB device that import chain, not the "
            f"{INT8_PATH.stat().st_size / 1024 ** 2:.0f} MiB INT8 file, is the larger cost."
        )
        print(f"baseline RSS      : {BASELINE_RSS / 1024 ** 2:>8.1f} MiB")
        print(f"privacy stack     : {MEMORY['privacy_stack_resident_bytes'] / 1024 ** 2:>8.1f} MiB resident")
    print(f"process RSS       : {MEMORY['process_rss_bytes'] / 1024 ** 2:>8.1f} MiB")
    print(f"process peak RSS  : {MEMORY['process_peak_rss_bytes'] / 1024 ** 2:>8.1f} MiB")
    print(f"system RAM        : {MEMORY['system_total_bytes'] / 1024 ** 3:>8.2f} GiB "
          f"({MEMORY['system_available_bytes'] / 1024 ** 3:.2f} GiB available)")
else:
    note("psutil is not installed; memory metrics are limited to artifact sizes")

print(f"INT8 model file   : {MEMORY['model_file_bytes'] / 1024 ** 2:>8.2f} MiB")

## 11. Save results

Everything below is written to disk. The notebook's own output is a convenience;
the files are the record, and `Research_Paper_Figures.ipynb` reads them.

In [ ]:
def write_json(name: str, payload) -> Path:
    path = RUN_DIR / name
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        stream.write(json.dumps(payload, indent=2, default=float) + "\n")
    return path


def write_csv(name: str, rows, fieldnames) -> Path | None:
    if not rows:
        return None
    path = RUN_DIR / name
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    return path


def write_jsonl(name: str, rows) -> Path | None:
    if not rows:
        return None
    path = RUN_DIR / name
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(json.dumps(row, default=float) + "\n")
    return path


FINISHED_UTC = datetime.now(timezone.utc)
BENCHMARK_CONFIG = {
    "run_id": RUN_ID,
    "mode": MODE,
    "smoke_test": SMOKE_TEST,
    "device_label": DEVICE_LABEL or None,
    "target_rows": TARGET_ROWS,
    "rows_scored": len(EVAL_ROWS),
    "benchmark_rows_full": BENCHMARK_ROWS,
    "smoke_rows": SMOKE_ROWS,
    "warmup_runs": WARMUP_RUNS,
    "latency_probe_rows": LATENCY_PROBE_ROWS,
    "pipeline_warmup_runs": PIPELINE_WARMUP_RUNS,
    "pipeline_latency_runs": PIPELINE_LATENCY_RUNS,
    "run_full_pipeline": RUN_FULL_PIPELINE,
    "run_ood_dev": RUN_OOD_DEV,
    "random_seed": RANDOM_SEED,
    "onnx_intra_op_threads": ONNX_INTRA_OP_THREADS,
    "subset": {
        "path": bench.SUBSET_PATH.relative_to(REPO_ROOT).as_posix(),
        "sha256": SUBSET_MANIFEST["fixture_sha256"],
        "selection": SUBSET_MANIFEST["selection"],
        "source": SUBSET_MANIFEST["source"],
        "coverage": SUBSET_MANIFEST["coverage"],
    },
    "started_utc": STARTED_UTC.isoformat(),
    "finished_utc": FINISHED_UTC.isoformat(),
    "duration_seconds": (FINISHED_UTC - STARTED_UTC).total_seconds(),
}

written = [
    write_json("system_info.json", SYSTEM_INFO),
    write_json("benchmark_config.json", BENCHMARK_CONFIG),
    write_json("artifact_manifest.json", ARTIFACT_MANIFEST),
    write_json("memory_metrics.json", MEMORY),
    write_json("pipeline_latency_summary.json", PIPELINE_SUMMARY),
]
if DEBERTA_LATENCY:
    written.append(write_json("deberta_latency_summary.json", DEBERTA_LATENCY))
if EVALUATION:
    payload = dict(EVALUATION)
    if OOD_RESULT:
        payload["ood_dev"] = {
            key: value for key, value in OOD_RESULT.items() if key != "parity"
        }
    written.append(write_json("evaluation_metrics.json", payload))

written.append(write_csv(
    "pipeline_latency.csv", PIPELINE_ROWS,
    ["utterance_index", "characters", "words", "transcript_merge_ms", "wake_word_ms",
     "inactivity_buffer_ms", "privacy_masking_ms", "digit_normalisation_ms",
     "text_path_total_ms"],
))
written.append(write_csv(
    "deberta_latency.csv",
    [{"row_id": record["row_id"], "characters": record["characters"],
      "windows": record["windows"], "latency_ms": record["latency_ms"]}
     for record in PARITY_RECORDS],
    ["row_id", "characters", "windows", "latency_ms"],
))
written.append(write_csv(
    "per_entity_metrics.csv", PER_ENTITY,
    ["view", "entity", "precision", "recall", "f1", "support", "predicted", "true_positives"],
))
written.append(write_jsonl("prediction_parity_data.jsonl", PARITY_RECORDS))
if OOD_RESULT:
    written.append(write_jsonl("ood_dev_parity_data.jsonl", OOD_RESULT["parity"]))

# One digest over every prediction, in fixture order: two devices that agree
# have the same string here, and no further comparison is needed to say so.
PREDICTIONS_DIGEST = hashlib.sha256(
    "\n".join(
        f"{record['row_id']}:{record['production_digest']}" for record in PARITY_RECORDS
    ).encode("utf-8")
).hexdigest()

written = [path for path in written if path is not None]
print(f"wrote {len(written)} files to {RUN_DIR.relative_to(REPO_ROOT).as_posix()}")
for path in written:
    print(f"  {path.name:<32} {path.stat().st_size:>9,} bytes")
print(f"\npredictions digest: {PREDICTIONS_DIGEST}")

In [ ]:
def megabytes(value):
    return f"{value / 1024 ** 2:.1f} MiB" if value else "unknown"


def latency_line(summary) -> str:
    """One markdown table cell. No pipe characters - they would split the row."""

    if not summary.get("count"):
        return "not measured"
    return (
        f"mean {summary['mean_ms']:.1f} ms, P50 {summary['p50_ms']:.1f}, "
        f"P95 {summary['p95_ms']:.1f}, P99 {summary['p99_ms']:.1f} (n={summary['count']})"
    )


benchmark_ok = bool(EVALUATION) and ARTIFACT_OK and not FAILURES
lines = [
    f"# CPU benchmark - {RUN_ID}",
    "",
    f"- **Completed successfully:** {'yes' if benchmark_ok else 'no - see warnings'}",
    f"- **Mode:** {MODE} ({len(EVAL_ROWS):,} of {len(benchmark_rows_all):,} benchmark rows)",
    f"- **Started / finished (UTC):** {STARTED_UTC.isoformat(timespec='seconds')} -> "
    f"{FINISHED_UTC.isoformat(timespec='seconds')} "
    f"({BENCHMARK_CONFIG['duration_seconds'] / 60:.1f} min)",
    "",
    "## Device",
    "",
    f"| Field | Value |",
    f"|---|---|",
    f"| OS | {SYSTEM_INFO['platform']} |",
    f"| Architecture | {SYSTEM_INFO['architecture']} ({SYSTEM_INFO['process_architecture']}) |",
    f"| CPU | {SYSTEM_INFO['cpu_model']} |",
    f"| Logical / physical cores | {SYSTEM_INFO['logical_cpus']} / {SYSTEM_INFO['physical_cpus']} |",
    f"| Board | {SYSTEM_INFO['board_model'] or 'not reported'} |",
    f"| RAM | {_gib(SYSTEM_INFO['total_ram_bytes'])} |",
    f"| Python | {SYSTEM_INFO['python']} |",
    f"| ONNX Runtime | {SYSTEM_INFO['onnxruntime']} |",
    "",
    "## Model artifact",
    "",
    f"- Path: `{artifact_files.get('model_int8.onnx', {}).get('path', 'missing')}`",
    f"- SHA-256: `{measured_sha}`",
    f"- Size: {megabytes(MEMORY.get('model_file_bytes'))}",
    f"- Identity verified against the release records: "
    f"{'yes' if ARTIFACT_MANIFEST['identity_verified'] else 'NO'}",
    f"- Benchmark subset SHA-256: `{SUBSET_MANIFEST['fixture_sha256']}`",
    f"- Predictions digest: `{PREDICTIONS_DIGEST}`",
    "",
]

if EVALUATION:
    production = EVALUATION["production_path"]
    lines += [
        "## Privacy model quality (production path)",
        "",
        "| Metric | Value |",
        "|---|---|",
        f"| Typed entity F1 | {production['typed_entity']['f1']:.4f} |",
        f"| Typed entity precision | {production['typed_entity']['precision']:.4f} |",
        f"| Typed entity recall | {production['typed_entity']['recall']:.4f} |",
        f"| Binary entity F1 | {production['binary_entity']['f1']:.4f} |",
        f"| Private character recall | {production['private_character_recall']:.4f} |",
        f"| Private character leakage | {production['private_character_leakage']:.4f} |",
        f"| Non-private overmasking | {production['non_private_overmasking_rate']:.4f} |",
        f"| Hard-negative FP rate | {production['hard_negative_false_positive_rate']:.4f} "
        f"({production['hard_negative_rows']} rows) |",
        f"| Model-attributed typed F1 | {EVALUATION['model_attributed']['typed_entity']['f1']:.4f} |",
        "",
    ]

if OOD_RESULT:
    released = OOD_RESULT.get("release_reference")
    lines += [
        "## Out-of-distribution cross-check (ood_dev, model-only)",
        "",
        f"- Cases scored: {OOD_RESULT['rows']}"
        f"{'' if OOD_RESULT['complete_fixture'] else ' (subset - smoke mode)'}",
        f"- Typed entity F1: {OOD_RESULT['metrics']['typed_entity']['f1']:.4f}",
    ]
    if released:
        lines.append(f"- Release reference: {released['typed_entity_f1']:.4f} "
                     f"(delta {OOD_RESULT['typed_f1_delta_vs_release']:+.4f})")
    lines.append("")

lines += [
    "## Latency",
    "",
    "| Stage | Statistics |",
    "|---|---|",
]
if DEBERTA_LATENCY:
    lines += [
        f"| DeBERTa full masking call | {latency_line(DEBERTA_LATENCY['end_to_end_masking_call'])} |",
        f"| - neural decode | {latency_line(DEBERTA_LATENCY['stages']['neural_decode'])} |",
        f"| - structured regex | {latency_line(DEBERTA_LATENCY['stages']['structured'])} |",
        f"| - span merge | {latency_line(DEBERTA_LATENCY['stages']['span_merge'])} |",
    ]
for name, entry in PIPELINE_STAGES.items():
    lines.append(
        f"| pipeline: {name} | "
        f"{latency_line(entry) if entry.get('available') else 'unavailable - ' + entry.get('reason', '')} |"
    )
if PIPELINE_SUMMARY.get("text_path_end_to_end", {}).get("count"):
    lines.append(f"| pipeline: text path end to end | "
                 f"{latency_line(PIPELINE_SUMMARY['text_path_end_to_end'])} |")
if "full_utterance_estimate_ms" in PIPELINE_SUMMARY:
    lines.append(f"| pipeline: assembled {AUDIO_SECONDS:.1f}s utterance | "
                 f"{PIPELINE_SUMMARY['full_utterance_estimate_ms']:.1f} ms "
                 f"({'complete' if PIPELINE_SUMMARY['complete'] else 'incomplete'}) |")
if DEBERTA_LATENCY.get("throughput_rows_per_second"):
    lines.append(f"| throughput | {DEBERTA_LATENCY['throughput_rows_per_second']:.2f} rows/s |")

lines += [
    "",
    "## Memory",
    "",
    f"- Process RSS: {megabytes(MEMORY.get('process_rss_bytes'))}",
    f"- Process peak RSS: {megabytes(MEMORY.get('process_peak_rss_bytes'))}",
    f"- Privacy stack resident (session + tokenizer + imports): "
    f"{megabytes(MEMORY.get('privacy_stack_resident_bytes'))}",
    f"- System RAM: {_gib(MEMORY.get('system_total_bytes'))}",
    f"- INT8 model on disk: {megabytes(MEMORY.get('model_file_bytes'))}",
    "",
    "## Warnings",
    "",
]
lines += [f"- {message}" for message in WARNINGS] or ["- none"]
lines += [
    "",
    "## Sections completed",
    "",
]
lines += [
    f"- {name}: {'yes' if done else 'no'}" for name, done in sorted(COMPLETED.items())
] or ["- none"]
lines.append("")

SUMMARY_PATH = RUN_DIR / "benchmark_summary.md"
SUMMARY_PATH.write_text("\n".join(lines), encoding="utf-8", newline="\n")
print(f"wrote {SUMMARY_PATH.relative_to(REPO_ROOT).as_posix()}\n")
print("\n".join(lines))

---

## Done

The folder printed above is the whole result. Commit it:

```bash
git add research_results/cpu_benchmarks/<run_id>
git commit -m "benchmark: <device> CPU results"
```

Then run `Research_Paper_Figures.ipynb` on any machine that has the repository.
It picks up every run under `research_results/cpu_benchmarks/`, plots the
latency and memory figures, and - once two devices have been recorded - the
workstation-versus-Raspberry-Pi comparison and the prediction parity check.

If a section reported *unavailable*, the reason is in the warnings list and in
`benchmark_summary.md`. Nothing was faked to fill a gap.